# Telephony and SIP

Parse a real-looking `INVITE`, put audio through a simulated phone line, and decode keypresses. Predict each result before you run the cell.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

invite = main.parse_sip_message(main.SAMPLE_INVITE)
offer = main.parse_sdp(invite['body'])
offer['payload_types'], main.negotiate_codec(offer), main.negotiate_codec(offer, ('PCMU',))

## RTP packets

How many packets does one second of G.711 produce, and how do `seq` and `timestamp` advance?

In [ ]:
codes = main.mulaw_encode(main.float_to_pcm16(0.3 * np.sin(2 * np.pi * 440 * np.arange(8000) / 8000)))
packets = main.packetize(codes.tobytes(), payload_type=0)
[(p['seq'], p['timestamp'], len(p['payload'])) for p in map(main.unpack_rtp_header, packets[:3])], main.rtp_bitrate_kbps()

## mu-law versus linear 8-bit

Try levels from 0 dBFS down to -50 dBFS. Where does linear PCM win?

In [ ]:
t = np.arange(8000) / 8000
for level_db in (0, -10, -20, -30, -40, -50):
    tone = 10 ** (level_db / 20) * np.sin(2 * np.pi * 1013 * t)
    print(f"{level_db:>4} dBFS  mu-law {main.snr_db(tone, main.mulaw_roundtrip(tone)):5.1f} dB   "
          f"linear {main.snr_db(tone, main.quantize_linear(tone, 8)):5.1f} dB")

## What the phone line removes

Change `f0` to a female voice (220 Hz) or a low male voice (90 Hz). Does the pitch survive the 300 Hz high-pass?

In [ ]:
f0 = 120.0
wide = main.synthetic_utterance(f0=f0)
phone = main.phone_channel(wide)
print('energy above 3.4 kHz before:', round(main.band_energy_fraction(wide, 16_000, 3400, 8000), 3))
print('pitch after the phone line:', round(main.autocorrelation_pitch(phone[:3200], 8000), 1), 'Hz')

## DTMF with Goertzel

Add noise and find where detection breaks.

In [ ]:
rng = np.random.default_rng(0)
for snr in (0, -10, -15, -20, -25):
    tone = main.dtmf_tone('5')
    noise = rng.standard_normal(len(tone)) * np.std(tone) / 10 ** (snr / 20)
    print(snr, 'dB SNR ->', main.detect_dtmf(main.mulaw_roundtrip(tone + noise)))